# Split a mesh {#ref_tutorials_split_mesh}

`MAPDL`{.interpreted-text role="bdg-mapdl"} `LSDYNA`{.interpreted-text
role="bdg-lsdyna"} `Fluent`{.interpreted-text role="bdg-fluent"}
`CFX`{.interpreted-text role="bdg-cfx"}

This tutorial shows how to split a mesh on a given property.

There are two approaches:

- `Use the split_mesh operator <ref_first_approach_split_mesh>`{.interpreted-text
  role="ref"} to split an already existing
  `MeshedRegion<ansys.dpf.core.meshed_region.MeshedRegion>`{.interpreted-text
  role="class"}.
- `Split the mesh scoping <ref_second_approach_split_mesh>`{.interpreted-text
  role="ref"} and create the split `MeshedRegion` objects.


# Import the necessary modules


In [ ]:
from ansys.dpf import core as dpf
from ansys.dpf.core import examples, operators as ops

# Define the mesh

For this tutorial, we get a `MeshedRegion` from a result file for each
solver. For more information see the
`ref_tutorials_get_mesh_from_result_file`{.interpreted-text role="ref"}
tutorial.

MAPDL --- use a multi-shell result file.


In [ ]:
result_file_path_1 = examples.find_multishells_rst()
model_1 = dpf.Model(data_sources=result_file_path_1)
meshed_region_1 = model_1.metadata.meshed_region

LS-DYNA


In [ ]:
result_file_path_2 = examples.download_d3plot_beam()
ds_2 = dpf.DataSources()
ds_2.set_result_file_path(filepath=result_file_path_2[0], key="d3plot")
ds_2.add_file_path(filepath=result_file_path_2[3], key="actunits")
model_2 = dpf.Model(data_sources=ds_2)
meshed_region_2 = model_2.metadata.meshed_region

Fluent


In [ ]:
result_file_path_3 = examples.download_fluent_axial_comp()["flprj"]
model_3 = dpf.Model(data_sources=result_file_path_3)
meshed_region_3 = model_3.metadata.meshed_region

CFX


In [ ]:
result_file_path_4 = examples.download_cfx_mixing_elbow()
model_4 = dpf.Model(data_sources=result_file_path_4)
meshed_region_4 = model_4.metadata.meshed_region

# First approach --- use the `split_mesh` operator {#ref_first_approach_split_mesh}

Use the
`split_mesh<ansys.dpf.core.operators.mesh.split_mesh.split_mesh>`{.interpreted-text
role="class"} operator to split an already existing `MeshedRegion` based
on a given property. Currently you can split a mesh by material
(`"mat"`) or element type (`"eltype"`).

The split mesh parts are stored in a
`MeshesContainer<ansys.dpf.core.meshes_container.MeshesContainer>`{.interpreted-text
role="class"}, ordered by `labels`. When using the `split_mesh`
operator, each split mesh part has two labels: a `"body"` label and a
label for the property used to split the mesh.

Here, we split the `MeshedRegion` by material.


In [ ]:
meshes_11 = ops.mesh.split_mesh(mesh=meshed_region_1, property="mat").eval()
print("Split mesh MAPDL:", meshes_11)

meshes_21 = ops.mesh.split_mesh(mesh=meshed_region_2, property="mat").eval()
print("Split mesh LSDYNA:", meshes_21)

meshes_31 = ops.mesh.split_mesh(mesh=meshed_region_3, property="mat").eval()
print("Split mesh Fluent:", meshes_31)

meshes_41 = ops.mesh.split_mesh(mesh=meshed_region_4, property="mat").eval()
print("Split mesh CFX:", meshes_41)

# Second approach --- split the scoping then build meshes {#ref_second_approach_split_mesh}

This approach:

1.  Uses the
    `split_on_property_type<ansys.dpf.core.operators.scoping.split_on_property_type.split_on_property_type>`{.interpreted-text
    role="class"} operator to split the mesh
    `Scoping<ansys.dpf.core.scoping.Scoping>`{.interpreted-text
    role="class"} on a given property. The split scoping is stored in a
    `ScopingsContainer<ansys.dpf.core.scopings_container.ScopingsContainer>`{.interpreted-text
    role="class"}, ordered by labels for the split property.
2.  Creates the split `MeshedRegion` objects using the
    `from_scopings<ansys.dpf.core.operators.mesh.from_scopings.from_scopings>`{.interpreted-text
    role="class"} operator for each `Scoping` of interest. The split
    parts are stored in a `MeshesContainer`.

Here, we split the mesh scoping by material and create a `MeshedRegion`
for all split `Scoping` objects in the `ScopingsContainer`.


In [ ]:
split_scoping_1 = ops.scoping.split_on_property_type(mesh=meshed_region_1, label1="mat").eval()
meshes_12 = ops.mesh.from_scopings(scopings_container=split_scoping_1, mesh=meshed_region_1).eval()
print("Split via scoping MAPDL:", meshes_12)

split_scoping_2 = ops.scoping.split_on_property_type(mesh=meshed_region_2, label1="mat").eval()
meshes_22 = ops.mesh.from_scopings(scopings_container=split_scoping_2, mesh=meshed_region_2).eval()
print("Split via scoping LSDYNA:", meshes_22)

split_scoping_3 = ops.scoping.split_on_property_type(mesh=meshed_region_3, label1="mat").eval()
meshes_32 = ops.mesh.from_scopings(scopings_container=split_scoping_3, mesh=meshed_region_3).eval()
print("Split via scoping Fluent:", meshes_32)

split_scoping_4 = ops.scoping.split_on_property_type(mesh=meshed_region_4, label1="mat").eval()
meshes_42 = ops.mesh.from_scopings(scopings_container=split_scoping_4, mesh=meshed_region_4).eval()
print("Split via scoping CFX:", meshes_42)